# 10 · Lectura de placas reales: EasyOCR frente a Tesseract

Se ejecuta en **Google Colab** (GPU T4). Mide la **lectura del texto** de la placa sobre **recortes de placas bolivianas reales** (anotadas a mano), que es lo que
faltaba: hasta ahora el OCR solo se había medido con 300 placas sintéticas y una foto real.

Compara tres lecturas sobre los mismos recortes:

1. **EasyOCR crudo** (lo que se usa en el servicio).
2. **EasyOCR + formato de A4** (`ml/src/placas.py: elegir_placa`: 4 dígitos + 3 letras; lo que no cumple el formato se rechaza).
3. **Tesseract** (el mismo motor que `Tesseract.js` en el navegador), con lista de caracteres y una sola línea / bloque.

Ya se midió **Tesseract.js en Node**: 3 de 10 placas exactas y error por carácter del 55 % en el mejor caso. Aquí se ve si EasyOCR lo hace mejor con los mismos recortes
(si sí, el problema es el motor; si no, son los recortes, que son pequeños: 58 a 174 px de ancho).

Sube: `ml.zip` y `recortes.zip` (PNG a color + `verdad.json`). Las placas son de vehículos de terceros: **no se suben a git** (I6).

In [ ]:
import os, zipfile, json
from google.colab import files

subidos = files.upload()     # ml.zip + recortes.zip
os.makedirs("/content/condohub-ia", exist_ok=True)
os.makedirs("/content/recortes", exist_ok=True)
for nombre in subidos:
    z = zipfile.ZipFile(nombre)
    z.extractall("/content/condohub-ia" if nombre.lower().startswith("ml") else "/content/recortes")
os.chdir("/content/condohub-ia")
verdad = json.load(open("/content/recortes/verdad.json", encoding="utf-8"))
print(len(verdad), "recortes:", [v["real"] for v in verdad])

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q easyocr pytesseract
import torch
print("GPU disponible:", torch.cuda.is_available())

## 1 · EasyOCR (crudo y con el formato de A4)

In [ ]:
import numpy as np
from PIL import Image
import easyocr
from ml.src import placas

lector = easyocr.Reader(["en"], gpu=torch.cuda.is_available())
PERMITIDOS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
reales, crudo, con_formato, confianzas = [], [], [], []
for v in verdad:
    imagen = Image.open(f"/content/recortes/{v['archivo']}").convert("RGB")
    bloques = [(t, float(c)) for _, t, c in lector.readtext(np.asarray(imagen), allowlist=PERMITIDOS)]
    placa, confianza = placas.elegir_placa(bloques)
    reales.append(v["real"])
    crudo.append("".join(t for t, _ in bloques))
    con_formato.append(placa or "")
    confianzas.append(confianza)
print(f"{'real':<9}{'EasyOCR crudo':<18}{'+ formato A4':<12}{'conf':>6}")
for r, c, f, k in zip(reales, crudo, con_formato, confianzas):
    print(f"{r:<9}{c:<18}{f or '(rechazada)':<12}{k:>6.2f}{'   ✓' if f == r else ''}")
m_crudo = placas.metricas_lectura(reales, crudo)
m_formato = placas.metricas_lectura(reales, con_formato)
print("\ncrudo:", {k: round(x, 3) for k, x in m_crudo.items()})
print("con formato A4:", {k: round(x, 3) for k, x in m_formato.items()})

## 2 · Tesseract (el motor de Tesseract.js)

Con lista de caracteres (dígitos y mayúsculas), a una línea (`--psm 7`) y como bloque (`--psm 6`), en el recorte original y ampliado ×3 y ×5 con autocontraste.

In [ ]:
import pytesseract
from PIL import ImageOps

def leer(imagen, psm):
    config = f"--psm {psm} -c tessedit_char_whitelist={PERMITIDOS}"
    return "".join(c for c in pytesseract.image_to_string(imagen, config=config) if c in PERMITIDOS)

tesseract = {}
for escala in (1, 3, 5):
    for psm in (7, 6):
        leidas = []
        for v in verdad:
            im = Image.open(f"/content/recortes/{v['archivo']}").convert("L")
            if escala > 1:
                im = ImageOps.autocontrast(im.resize((im.width * escala, im.height * escala), Image.LANCZOS))
            leidas.append(leer(im, psm))
        m = placas.metricas_lectura(reales, leidas)
        tesseract[f"x{escala}_psm{psm}"] = {**m}
        print(f"Tesseract x{escala} psm {psm}: exactas {m['placa_completa'] * len(reales):.0f}/{len(reales)} | precisión por carácter {m['precision_por_caracter']:.3f}")

## 3 · Guardar y descargar

Solo métricas: ni las imágenes ni las placas (reales o leídas) se guardan, porque son de vehículos de terceros (I6).

In [ ]:
from google.colab import files
from ml.src.paths import dir_experimento

destino = dir_experimento("placas_reales_ocr")
resultado = {
    "recortes": len(verdad), "tam_recortes_px": [v["tam"] for v in verdad],
    "easyocr_crudo": {**m_crudo},
    "easyocr_con_formato_A4": {**m_formato, "confianzas": confianzas},
    "tesseract": tesseract,
    "tesseract_js_en_node": {"mejor": "x5 psm 6", "placas_exactas": "3/10", "error_por_caracter": 0.551},
}
(destino / "metrics.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8")
files.download(str(destino / "metrics.json"))